# Lab 09: Inference Optimization & Cost Management

## Overview
In this lab you will optimize inference for cost and latency. You will compare provisioned vs on-demand throughput modes, implement batch inference for large-scale processing, build a semantic cache using embeddings to eliminate redundant API calls, optimize token usage through prompt engineering and model tiering, benchmark latency across models, and perform cost analysis for production workloads.

## Learning Objectives
- Understand on-demand vs provisioned throughput pricing models and when to use each
- Implement batch inference jobs for cost-efficient large-scale processing
- Build semantic caching with configurable similarity thresholds to reduce API costs
- Optimize token usage through concise prompting and model tiering strategies
- Benchmark latency across different models and configurations
- Perform cost analysis for production workloads at scale

## Exam Domain
**Optimizing Performance & Inference (24%)** — This lab covers Task 3.2 (optimize inference performance and cost), focusing on throughput modes, batch processing, caching strategies, token optimization, latency benchmarking, and cost modeling for production deployments.

## Architecture Diagram
![Lab 09 Architecture](../assets/diagrams/lab-09-inference-optimization.png)

In [ ]:
%pip install boto3 numpy --quiet

In [ ]:
import boto3, json, os, time
import numpy as np

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

bedrock_runtime = session.client("bedrock-runtime")
bedrock = session.client("bedrock")
s3 = session.client("s3")
sts = session.client("sts")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
BUCKET = f"aws-genai-lab-{ACCOUNT_ID}"
BEDROCK_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/genai-lab-bedrock-role"

print(f"Account:      {ACCOUNT_ID}")
print(f"S3 bucket:    {BUCKET}")
print(f"Bedrock role: {BEDROCK_ROLE_ARN}")

In [ ]:
# Helper: unified Converse API wrapper for any Bedrock text model
def invoke_converse(model_id, prompt, max_tokens=256, temperature=0.7):
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": temperature}
    )
    return response


# Helper: Titan Embeddings V2
def get_embedding(text):
    """Get embedding from Titan Embeddings V2."""
    response = bedrock_runtime.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        contentType="application/json",
        body=json.dumps({"inputText": text})
    )
    return json.loads(response["body"].read())["embedding"]


# Helper: cosine similarity
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Pricing per 1K tokens (USD) — approximate as of 2025
PRICING = {
    "anthropic.claude-3-5-sonnet-20241022-v2:0": {"input": 0.003, "output": 0.015, "name": "Claude 3.5 Sonnet"},
    "anthropic.claude-3-5-haiku-20241022-v1:0": {"input": 0.0008, "output": 0.004, "name": "Claude 3.5 Haiku"},
    "amazon.titan-text-express-v1": {"input": 0.0002, "output": 0.0006, "name": "Titan Text Express"},
}

print("Helper functions and pricing table defined.")

## A. On-Demand vs Provisioned Throughput

Amazon Bedrock offers two throughput modes for inference. Choosing the right mode is a key cost and performance optimization decision.

| Feature | On-Demand | Provisioned |
|---------|-----------|-------------|
| Pricing | Per token (input + output) | Per hour (model units) |
| Latency | Variable (shared capacity) | Consistent (dedicated capacity) |
| Throughput | Shared, subject to throttling | Guaranteed minimum |
| Commitment | None — pay as you go | 1-month or 6-month term |
| Best for | Dev/test, variable/unpredictable load | Production, predictable high-volume load |
| Cost efficiency | Lower at small scale | Lower at high scale (break-even ~50K+ calls/hr) |

**On-demand** is the default mode — you pay per token with no upfront commitment. It is ideal for development, testing, and workloads with variable traffic patterns. The tradeoff is that latency can vary under high load because capacity is shared across customers.

**Provisioned throughput** reserves dedicated capacity measured in **model units**. Each model unit guarantees a specific number of input/output tokens per minute. You pay a fixed hourly rate regardless of actual usage, which makes it cost-effective only when your sustained volume exceeds the break-even point.

> **Exam tip:** The exam tests whether you know when to recommend provisioned throughput. The answer is: predictable, high-volume production workloads where consistent latency matters and the hourly cost is lower than equivalent on-demand token charges.

In [ ]:
# Check which foundation models support provisioned throughput
foundation_models = bedrock.list_foundation_models()["modelSummaries"]

provisioned_models = [
    m for m in foundation_models
    if "PROVISIONED" in m.get("inferenceTypesSupported", [])
]

print(f"Models supporting provisioned throughput: {len(provisioned_models)} of {len(foundation_models)}\n")
print(f"{'Model ID':<55} {'Provider':<15} {'Status'}")
print("-" * 90)
for m in provisioned_models[:10]:
    print(f"{m['modelId']:<55} {m['providerName']:<15} {m['modelLifecycle']['status']}")
if len(provisioned_models) > 10:
    print(f"... and {len(provisioned_models) - 10} more")

In [ ]:
# Provisioned throughput creation (DO NOT run — charges ~$50+/hr depending on model)
# This is commented out to avoid incurring significant costs.
# In production, you would create provisioned throughput like this:

# response = bedrock.create_provisioned_model_throughput(
#     modelUnits=1,
#     provisionedModelName="genai-lab-provisioned",
#     modelId="anthropic.claude-3-5-sonnet-20241022-v2:0",
#     commitmentDuration="OneMonth"  # or "SixMonths" for lower rate
# )
# provisioned_arn = response["provisionedModelArn"]
# print(f"Provisioned throughput ARN: {provisioned_arn}")
#
# # To use provisioned throughput, pass the ARN as the modelId:
# response = bedrock_runtime.converse(
#     modelId=provisioned_arn,
#     messages=[{"role": "user", "content": [{"text": "Hello"}]}],
#     inferenceConfig={"maxTokens": 100}
# )
#
# # To delete when no longer needed:
# bedrock.delete_provisioned_model_throughput(
#     provisionedModelId=provisioned_arn
# )

print("Provisioned throughput code shown above (commented out to avoid charges).")
print("Key points:")
print("  - create_provisioned_model_throughput() reserves dedicated capacity")
print("  - Use the returned ARN as modelId in converse() calls")
print("  - Charges apply for the full commitment period (1 or 6 months)")
print("  - delete_provisioned_model_throughput() releases the reservation")

## B. Batch Inference

Batch inference processes multiple prompts in a single asynchronous job rather than making individual API calls. Bedrock's `create_model_invocation_job` API reads a JSONL file from S3, invokes the model for each record, and writes results back to S3.

**When to use batch inference:**
- Large-scale processing (hundreds or thousands of prompts)
- Non-time-sensitive workloads (results needed in minutes/hours, not seconds)
- Cost optimization — batch inference costs approximately **50% less** than equivalent on-demand calls

**How it works:**
1. Prepare a JSONL input file where each line contains a `recordId` and `modelInput`
2. Upload the file to S3
3. Create a model invocation job pointing to the input file
4. Bedrock processes all records asynchronously and writes output to S3
5. Download and parse the results

> **Exam tip:** Batch inference is the correct answer when the question describes a workload that needs to process large volumes of data cost-effectively and does not require real-time responses.

In [ ]:
# Prepare batch input file — JSONL format with recordId and modelInput per line
batch_prompts = [
    {
        "recordId": "1",
        "modelInput": {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 200,
            "messages": [{"role": "user", "content": "What is Amazon Bedrock?"}]
        }
    },
    {
        "recordId": "2",
        "modelInput": {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 200,
            "messages": [{"role": "user", "content": "What is Amazon SageMaker?"}]
        }
    },
    {
        "recordId": "3",
        "modelInput": {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 200,
            "messages": [{"role": "user", "content": "What is a foundation model?"}]
        }
    },
    {
        "recordId": "4",
        "modelInput": {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 200,
            "messages": [{"role": "user", "content": "What is retrieval-augmented generation (RAG)?"}]
        }
    },
    {
        "recordId": "5",
        "modelInput": {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 200,
            "messages": [{"role": "user", "content": "What are embeddings used for in AI?"}]
        }
    },
]

# Convert to JSONL and upload to S3
batch_key = "batch/input.jsonl"
body = "\n".join(json.dumps(p) for p in batch_prompts)
s3.put_object(Bucket=BUCKET, Key=batch_key, Body=body.encode("utf-8"))

print(f"Uploaded {len(batch_prompts)} prompts to s3://{BUCKET}/{batch_key}")
print(f"File size: {len(body):,} bytes")
print(f"\nSample record:")
print(json.dumps(batch_prompts[0], indent=2))

In [ ]:
# Create a batch inference job
BATCH_JOB_NAME = f"genai-lab-batch-{int(time.time())}"
BATCH_OUTPUT_PREFIX = "batch/output/"

try:
    job = bedrock.create_model_invocation_job(
        jobName=BATCH_JOB_NAME,
        roleArn=BEDROCK_ROLE_ARN,
        modelId="anthropic.claude-3-5-sonnet-20241022-v2:0",
        inputDataConfig={
            "s3InputDataConfig": {
                "s3Uri": f"s3://{BUCKET}/{batch_key}",
                "s3InputFormat": "JSONL"
            }
        },
        outputDataConfig={
            "s3OutputDataConfig": {
                "s3Uri": f"s3://{BUCKET}/{BATCH_OUTPUT_PREFIX}"
            }
        }
    )
    BATCH_JOB_ARN = job["jobArn"]
    print(f"Batch job created: {BATCH_JOB_NAME}")
    print(f"Job ARN: {BATCH_JOB_ARN}")

except Exception as e:
    BATCH_JOB_ARN = None
    print(f"Could not create batch job: {e}")
    print("This is expected if your IAM role lacks bedrock:CreateModelInvocationJob permission.")
    print("The batch job concept is still testable — see the exam prep section.")

In [ ]:
# Poll for batch job completion and download results
if BATCH_JOB_ARN:
    print("Polling for batch job completion (this may take several minutes)...")
    max_wait = 600  # 10 minutes
    poll_interval = 15
    elapsed = 0

    while elapsed < max_wait:
        status_response = bedrock.get_model_invocation_job(jobIdentifier=BATCH_JOB_ARN)
        status = status_response["status"]
        print(f"  [{elapsed}s] Status: {status}")

        if status == "Completed":
            print(f"\nBatch job completed successfully!")
            break
        elif status in ("Failed", "Stopped"):
            print(f"\nBatch job {status}.")
            if "message" in status_response:
                print(f"Message: {status_response['message']}")
            break

        time.sleep(poll_interval)
        elapsed += poll_interval
    else:
        print(f"\nTimed out after {max_wait}s. Job may still be running.")

    # Download and display results if completed
    if status == "Completed":
        output_uri = status_response.get("outputDataConfig", {}).get("s3OutputDataConfig", {}).get("s3Uri", "")
        print(f"\nOutput location: {output_uri}")

        # List output objects
        output_objects = s3.list_objects_v2(
            Bucket=BUCKET, Prefix=BATCH_OUTPUT_PREFIX
        ).get("Contents", [])

        for obj in output_objects:
            if obj["Key"].endswith(".jsonl.out"):
                data = s3.get_object(Bucket=BUCKET, Key=obj["Key"])["Body"].read().decode()
                print(f"\nResults from {obj['Key']}:")
                for line in data.strip().split("\n"):
                    record = json.loads(line)
                    record_id = record.get("recordId", "?")
                    output = record.get("modelOutput", {})
                    if "content" in output:
                        text = output["content"][0]["text"][:100]
                    else:
                        text = str(output)[:100]
                    print(f"  Record {record_id}: {text}...")
else:
    print("Batch job was not created. Skipping result download.")
    print("In a production scenario, batch results would be available in S3 as JSONL output.")

## C. Semantic Caching

Semantic caching reduces API costs and latency by storing previous responses and returning them when a sufficiently similar query arrives — even if the wording is different. Unlike exact-match caching, semantic caching uses **embeddings** to measure similarity between queries, so "What is S3?" and "Explain Amazon S3" would be a cache hit.

**How it works:**
1. When a query arrives, compute its embedding
2. Compare against all cached query embeddings using cosine similarity
3. If the best match exceeds the **similarity threshold**, return the cached response (cache hit)
4. Otherwise, call the model, store the query embedding + response in the cache (cache miss)

**Threshold tuning tradeoffs:**
- **High threshold (e.g., 0.95)** — fewer false hits, but only nearly identical queries match; minimal cost savings
- **Low threshold (e.g., 0.80)** — more cache hits and cost savings, but risk of returning irrelevant cached responses
- **Sweet spot (0.85-0.92)** — balances cost savings with response relevance for most use cases

In [ ]:
# Semantic cache implementation using Titan Embeddings
class SemanticCache:
    """Embedding-based semantic cache for LLM responses."""

    def __init__(self, similarity_threshold=0.90):
        self.threshold = similarity_threshold
        self.cache = []  # List of {"query": str, "embedding": list, "response": str}
        self.stats = {"hits": 0, "misses": 0}

    def _find_similar(self, query_embedding):
        """Find the most similar cached query. Returns (similarity, cached_entry) or (0, None)."""
        if not self.cache:
            return 0.0, None

        best_sim = 0.0
        best_entry = None
        for entry in self.cache:
            sim = cosine_similarity(query_embedding, entry["embedding"])
            if sim > best_sim:
                best_sim = sim
                best_entry = entry

        return best_sim, best_entry

    def query(self, prompt, model_id="anthropic.claude-3-5-sonnet-20241022-v2:0", max_tokens=200):
        """Query with semantic caching. Returns (response_text, cache_hit, similarity)."""
        # Step 1: Compute embedding for the query
        query_embedding = get_embedding(prompt)

        # Step 2: Check cache for similar query
        similarity, cached_entry = self._find_similar(query_embedding)

        if similarity >= self.threshold:
            # Cache hit
            self.stats["hits"] += 1
            return cached_entry["response"], True, similarity

        # Cache miss — call the model
        response = bedrock_runtime.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": max_tokens}
        )
        response_text = response["output"]["message"]["content"][0]["text"]

        # Store in cache
        self.cache.append({
            "query": prompt,
            "embedding": query_embedding,
            "response": response_text
        })
        self.stats["misses"] += 1

        return response_text, False, similarity

    def get_stats(self):
        total = self.stats["hits"] + self.stats["misses"]
        hit_rate = self.stats["hits"] / total * 100 if total > 0 else 0
        return {**self.stats, "total": total, "hit_rate": f"{hit_rate:.1f}%"}


cache = SemanticCache(similarity_threshold=0.90)
print(f"Semantic cache initialized (threshold: {cache.threshold})")
print(f"Cache size: {len(cache.cache)} entries")

In [ ]:
# Demonstrate semantic caching — cache miss followed by cache hit
test_queries = [
    ("What is Amazon Bedrock?", "Original query"),
    ("Explain Amazon Bedrock to me", "Semantically similar — should be cache hit"),
    ("Tell me about Amazon Bedrock service", "Another similar phrasing — should be cache hit"),
    ("What is Amazon SageMaker?", "Different topic — should be cache miss"),
    ("What is AWS SageMaker used for?", "Similar to previous — should be cache hit"),
]

print(f"{'Query':<45} {'Result':<12} {'Similarity':<12} {'Time'}")
print("-" * 95)

for query, description in test_queries:
    start = time.time()
    response_text, cache_hit, similarity = cache.query(query)
    elapsed = time.time() - start

    result = "HIT" if cache_hit else "MISS"
    print(f"{query:<45} {result:<12} {similarity:.4f}       {elapsed:.3f}s")

print(f"\n--- Cache Statistics ---")
stats = cache.get_stats()
print(f"Hits: {stats['hits']}  |  Misses: {stats['misses']}  |  Hit Rate: {stats['hit_rate']}")
print(f"Cache entries: {len(cache.cache)}")
print(f"\nNote: Cache hits return in ~0.1-0.3s (embedding only), misses take ~1-3s (embedding + LLM call).")

## D. Token Optimization

Every token costs money. Optimizing token usage is the most direct way to reduce inference costs. Two key strategies:

1. **Prompt engineering for token efficiency** — concise prompts produce fewer input tokens and often lead to more focused (shorter) output tokens
2. **Model tiering** — use a cheaper/faster model for simple tasks and reserve expensive models for complex tasks

Since Bedrock charges per token (input and output separately), reducing token counts directly lowers cost. A verbose prompt that produces the same quality answer as a concise prompt is wasting money at scale.

In [ ]:
# Compare verbose vs concise prompts — measure token usage and cost
verbose = (
    "I would like you to please provide me with a detailed explanation of "
    "what Amazon Bedrock is, including all of its features and capabilities, "
    "and also tell me about the different foundation models that are available "
    "through the service and how they can be used for various AI tasks."
)
concise = "What is Amazon Bedrock? List key features."

MODEL_ID = "anthropic.claude-3-5-sonnet-20241022-v2:0"
pricing = PRICING[MODEL_ID]

print(f"{'Prompt':<10} {'Input Tok':<12} {'Output Tok':<12} {'Total Tok':<12} {'Est. Cost'}")
print("-" * 65)

for label, prompt in [("Verbose", verbose), ("Concise", concise)]:
    response = bedrock_runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 200}
    )
    usage = response["usage"]
    input_tok = usage["inputTokens"]
    output_tok = usage["outputTokens"]
    total_tok = input_tok + output_tok
    cost = (input_tok / 1000 * pricing["input"]) + (output_tok / 1000 * pricing["output"])
    print(f"{label:<10} {input_tok:<12} {output_tok:<12} {total_tok:<12} ${cost:.5f}")

print(f"\nThe concise prompt achieves the same result with fewer tokens.")
print(f"At 1M requests/month, even small per-request savings compound significantly.")

In [ ]:
# Model tiering strategy — route tasks to the cheapest capable model
# Simple classification task: Haiku is sufficient
# Complex reasoning task: Sonnet is needed

simple_task = "Classify this as positive, negative, or neutral: 'The product arrived on time and works great.'"
complex_task = (
    "A company processes 10M documents/month. Each document requires summarization (200 tokens avg output) "
    "and entity extraction (50 tokens avg output). Compare the cost of using Claude Sonnet vs Haiku for each "
    "task and recommend an optimal tiering strategy."
)

tier_models = [
    ("anthropic.claude-3-5-haiku-20241022-v1:0", "Haiku (fast/cheap)"),
    ("anthropic.claude-3-5-sonnet-20241022-v2:0", "Sonnet (capable/expensive)"),
]

print("=== Simple Task: Sentiment Classification ===")
print(f"Prompt: {simple_task[:80]}...\n")

for model_id, name in tier_models:
    start = time.time()
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": simple_task}]}],
        inferenceConfig={"maxTokens": 50}
    )
    elapsed = time.time() - start
    usage = response["usage"]
    text = response["output"]["message"]["content"][0]["text"]
    p = PRICING[model_id]
    cost = (usage["inputTokens"] / 1000 * p["input"]) + (usage["outputTokens"] / 1000 * p["output"])
    print(f"  {name:<30} {elapsed:.2f}s  tokens={usage['inputTokens']+usage['outputTokens']}  cost=${cost:.6f}")
    print(f"    Response: {text[:80]}")

print(f"\n=== Complex Task: Cost Analysis ===")
print(f"Prompt: {complex_task[:80]}...\n")

for model_id, name in tier_models:
    start = time.time()
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": complex_task}]}],
        inferenceConfig={"maxTokens": 300}
    )
    elapsed = time.time() - start
    usage = response["usage"]
    text = response["output"]["message"]["content"][0]["text"]
    p = PRICING[model_id]
    cost = (usage["inputTokens"] / 1000 * p["input"]) + (usage["outputTokens"] / 1000 * p["output"])
    print(f"  {name:<30} {elapsed:.2f}s  tokens={usage['inputTokens']+usage['outputTokens']}  cost=${cost:.6f}")
    print(f"    Response (first 120 chars): {text[:120]}...")

print(f"\nTiering strategy: Use Haiku for simple tasks (classification, extraction, routing)")
print(f"and Sonnet for complex tasks (analysis, reasoning, generation). This can cut costs 50-80%.")

## E. Latency Benchmarking

Latency is critical for user-facing applications. Different models have different latency profiles based on model size, architecture, and infrastructure. This section benchmarks end-to-end response time across three models to help inform model selection for latency-sensitive workloads.

Factors that affect inference latency:
- **Model size** — smaller models (Haiku, Titan Express) respond faster than larger models (Sonnet)
- **Output length** — more output tokens = longer generation time
- **Throughput mode** — provisioned throughput provides more consistent latency than on-demand
- **Region** — choosing a region closer to your users reduces network latency

In [ ]:
# Latency benchmark across models
models = [
    ("anthropic.claude-3-5-sonnet-20241022-v2:0", "Claude 3.5 Sonnet"),
    ("anthropic.claude-3-5-haiku-20241022-v1:0", "Claude 3.5 Haiku"),
    ("amazon.titan-text-express-v1", "Titan Text Express"),
]

prompt = "What is Amazon Bedrock?"
NUM_RUNS = 3  # Average over multiple runs for more stable results
results = []

print(f"Benchmarking {NUM_RUNS} runs per model with prompt: \"{prompt}\"")
print(f"Max tokens: 100\n")

for model_id, name in models:
    latencies = []
    last_usage = None

    for run in range(NUM_RUNS):
        start = time.time()
        response = bedrock_runtime.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": 100}
        )
        elapsed = time.time() - start
        latencies.append(elapsed)
        last_usage = response["usage"]

    avg_latency = sum(latencies) / len(latencies)
    min_latency = min(latencies)
    max_latency = max(latencies)

    results.append({
        "model": name,
        "model_id": model_id,
        "avg_latency": avg_latency,
        "min_latency": min_latency,
        "max_latency": max_latency,
        "input_tokens": last_usage["inputTokens"],
        "output_tokens": last_usage["outputTokens"],
    })

# Display results
print(f"{'Model':<25} {'Avg (s)':<10} {'Min (s)':<10} {'Max (s)':<10} {'In Tok':<10} {'Out Tok'}")
print("-" * 80)
for r in results:
    print(
        f"{r['model']:<25} {r['avg_latency']:<10.2f} {r['min_latency']:<10.2f} "
        f"{r['max_latency']:<10.2f} {r['input_tokens']:<10} {r['output_tokens']}"
    )

# Identify fastest model
fastest = min(results, key=lambda x: x["avg_latency"])
print(f"\nFastest model: {fastest['model']} ({fastest['avg_latency']:.2f}s avg)")

In [ ]:
# Benchmark the effect of max_tokens on latency (using Haiku for speed)
HAIKU_ID = "anthropic.claude-3-5-haiku-20241022-v1:0"
token_limits = [50, 100, 200, 500]

print(f"Effect of maxTokens on latency (model: Claude 3.5 Haiku)")
print(f"Prompt: \"{prompt}\"\n")
print(f"{'maxTokens':<12} {'Latency (s)':<14} {'Input Tok':<12} {'Output Tok':<12} {'Actual < Max?'}")
print("-" * 65)

for max_tok in token_limits:
    start = time.time()
    response = bedrock_runtime.converse(
        modelId=HAIKU_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tok}
    )
    elapsed = time.time() - start
    usage = response["usage"]
    capped = "Yes" if usage["outputTokens"] < max_tok else "No (hit limit)"
    print(f"{max_tok:<12} {elapsed:<14.2f} {usage['inputTokens']:<12} {usage['outputTokens']:<12} {capped}")

print(f"\nNote: Setting maxTokens higher than needed does not increase latency")
print(f"if the model naturally finishes earlier. But it can increase cost if the")
print(f"model generates more tokens to fill the allowance.")

## F. Cost Analysis

This section models the cost of a hypothetical production workload — **1 million requests per month** — across different models and optimization strategies. Understanding cost at scale is essential for the exam and for real-world deployments.

We will calculate:
1. **Per-request cost** by model (using actual token counts from our benchmarks)
2. **Monthly cost at 1M requests** by model
3. **Cost savings from optimization strategies** (tiering, caching, batch)

In [ ]:
# Cost analysis for 1M requests/month using actual benchmark token counts
MONTHLY_REQUESTS = 1_000_000

print(f"=== Cost Projection: {MONTHLY_REQUESTS:,} requests/month ===\n")

# Use token counts from our latency benchmark
print(f"{'Model':<25} {'In Tok':<10} {'Out Tok':<10} {'Cost/Req':<14} {'Monthly Cost':<14} {'Annual Cost'}")
print("-" * 95)

cost_results = []
for r in results:
    p = PRICING[r["model_id"]]
    cost_per_req = (r["input_tokens"] / 1000 * p["input"]) + (r["output_tokens"] / 1000 * p["output"])
    monthly_cost = cost_per_req * MONTHLY_REQUESTS
    annual_cost = monthly_cost * 12
    cost_results.append({
        "model": r["model"],
        "cost_per_req": cost_per_req,
        "monthly": monthly_cost,
        "annual": annual_cost,
    })
    print(
        f"{r['model']:<25} {r['input_tokens']:<10} {r['output_tokens']:<10} "
        f"${cost_per_req:<13.6f} ${monthly_cost:<13,.0f} ${annual_cost:,.0f}"
    )

# Calculate savings from optimization strategies
sonnet_monthly = cost_results[0]["monthly"]  # Claude Sonnet as baseline
haiku_monthly = cost_results[1]["monthly"]

print(f"\n=== Optimization Strategies ===\n")

# Strategy 1: Model tiering (70% Haiku, 30% Sonnet)
tiered_monthly = (0.70 * haiku_monthly) + (0.30 * sonnet_monthly)
tiered_savings = (1 - tiered_monthly / sonnet_monthly) * 100
print(f"1. Model Tiering (70% Haiku / 30% Sonnet):")
print(f"   Monthly: ${tiered_monthly:,.0f}  |  Savings vs all-Sonnet: {tiered_savings:.0f}%")

# Strategy 2: Semantic caching (assume 40% cache hit rate)
cache_hit_rate = 0.40
cached_monthly = sonnet_monthly * (1 - cache_hit_rate)
cache_savings = cache_hit_rate * 100
print(f"\n2. Semantic Caching (40% hit rate, Sonnet):")
print(f"   Monthly: ${cached_monthly:,.0f}  |  Savings: {cache_savings:.0f}%")

# Strategy 3: Batch inference (50% discount)
batch_monthly = sonnet_monthly * 0.50
print(f"\n3. Batch Inference (50% discount, Sonnet):")
print(f"   Monthly: ${batch_monthly:,.0f}  |  Savings: 50%")

# Strategy 4: Combined (tiering + caching)
combined_monthly = tiered_monthly * (1 - cache_hit_rate)
combined_savings = (1 - combined_monthly / sonnet_monthly) * 100
print(f"\n4. Combined (Tiering + Caching):")
print(f"   Monthly: ${combined_monthly:,.0f}  |  Savings vs all-Sonnet: {combined_savings:.0f}%")

## Cleanup

Remove batch inference input/output files from S3 and stop any running batch jobs.

In [ ]:
# Clean up S3 batch data
try:
    # Delete batch input
    s3.delete_object(Bucket=BUCKET, Key=batch_key)
    print(f"Deleted: s3://{BUCKET}/{batch_key}")

    # Delete batch output objects
    batch_objects = s3.list_objects_v2(
        Bucket=BUCKET, Prefix="batch/"
    ).get("Contents", [])
    for obj in batch_objects:
        s3.delete_object(Bucket=BUCKET, Key=obj["Key"])
        print(f"Deleted: s3://{BUCKET}/{obj['Key']}")
except Exception as e:
    print(f"Cleanup error: {e}")

# Stop batch job if still running
if BATCH_JOB_ARN:
    try:
        job_status = bedrock.get_model_invocation_job(jobIdentifier=BATCH_JOB_ARN)
        if job_status["status"] in ("InProgress", "Submitted"):
            bedrock.stop_model_invocation_job(jobIdentifier=BATCH_JOB_ARN)
            print(f"Stopped batch job: {BATCH_JOB_NAME}")
        else:
            print(f"Batch job already {job_status['status']}")
    except Exception as e:
        print(f"Could not stop batch job: {e}")

print("\nCleanup complete.")

## Key Takeaways

1. **On-demand throughput is ideal for development and variable workloads, while provisioned throughput guarantees consistent latency for predictable production traffic** — the right choice depends on volume, latency requirements, and cost sensitivity at your expected scale
2. **Batch inference reduces cost by ~50% compared to on-demand for large-scale non-real-time processing** — the `create_model_invocation_job` API handles asynchronous processing of JSONL datasets stored in S3, making it the best choice for bulk document processing, data enrichment, and offline analysis
3. **Semantic caching eliminates redundant API calls by matching similar queries using embeddings** — a well-tuned similarity threshold (0.85-0.92) can achieve 30-50% cache hit rates in production, directly reducing both cost and latency for repeat or similar queries
4. **Token optimization through concise prompting and model tiering is the most impactful cost lever** — routing simple tasks to Haiku and reserving Sonnet for complex reasoning can reduce costs by 50-80% with minimal quality impact on the simpler tasks
5. **Latency varies significantly across models and should be benchmarked for your specific use case** — Haiku and Titan Express are 2-5x faster than Sonnet for equivalent prompts, making them better choices for latency-sensitive applications where top-tier reasoning is not required

## Key Concepts

| Concept | Definition |
|---------|-----------|
| On-Demand Throughput | The default Bedrock pricing mode where you pay per input and output token with no upfront commitment, offering flexibility for variable workloads but with shared capacity and variable latency |
| Provisioned Throughput | A reserved capacity mode that guarantees a specific number of model units (tokens/minute) for a fixed hourly rate, providing consistent latency and throughput for production workloads with 1-month or 6-month commitments |
| Batch Inference | Asynchronous processing of multiple prompts in a single job using `create_model_invocation_job`, reading JSONL input from S3 and writing results back to S3, with approximately 50% cost savings compared to on-demand inference |
| Semantic Cache | A caching layer that uses embedding similarity (rather than exact string matching) to identify and return previously computed responses for semantically equivalent queries, reducing both API cost and latency |
| Token Optimization | Strategies to minimize token usage including concise prompt engineering, setting appropriate maxTokens limits, and structuring prompts to elicit focused responses — directly reduces per-request cost |
| Model Tiering | A cost optimization pattern that routes requests to different models based on task complexity — using cheaper/faster models (e.g., Haiku) for simple tasks and reserving expensive/capable models (e.g., Sonnet) for complex reasoning |
| Latency | The end-to-end time from sending a request to receiving the complete response, influenced by model size, output length, throughput mode, and network distance — a critical metric for user-facing applications |

## Exam Preparation — Common Exam Question Patterns

**Q: When should you use provisioned throughput instead of on-demand in Amazon Bedrock?**
A: Use provisioned throughput when your application has predictable, high-volume traffic that requires consistent low latency. Provisioned throughput reserves dedicated model capacity measured in model units, guaranteeing a minimum tokens-per-minute rate. It is cost-effective when sustained usage exceeds the break-even point where the fixed hourly rate is cheaper than equivalent per-token on-demand charges. On-demand is better for development, testing, and unpredictable workloads.

**Q: How does batch inference in Bedrock reduce costs compared to on-demand inference?**
A: Batch inference uses `create_model_invocation_job` to process multiple prompts asynchronously from a JSONL file in S3, offering approximately 50% cost savings over on-demand inference. The tradeoff is that results are not real-time — the job runs asynchronously and writes output back to S3. Batch is ideal for large-scale processing tasks like document summarization, data enrichment, and content classification where immediate responses are not required.

**Q: A company wants to reduce inference costs for a chatbot that receives many similar questions. What optimization would you recommend?**
A: Implement semantic caching using embeddings. When a user query arrives, compute its embedding and compare it against cached query embeddings using cosine similarity. If a cached query exceeds the similarity threshold (typically 0.85-0.92), return the cached response without calling the model. This eliminates redundant API calls for semantically equivalent questions, reducing both cost and latency. For a chatbot with repetitive questions, cache hit rates of 30-50% are achievable.

**Q: How would you design a cost-optimized inference pipeline that handles both simple and complex tasks?**
A: Implement a model tiering strategy with a lightweight classifier or rule-based router. Route simple tasks (classification, extraction, yes/no questions) to a fast, inexpensive model like Claude Haiku, and route complex tasks (multi-step reasoning, analysis, creative generation) to a capable model like Claude Sonnet. This can reduce costs by 50-80% compared to using the expensive model for all tasks. Combine with semantic caching for further savings on repetitive queries and batch inference for non-real-time workloads.

**Q: What factors should you consider when benchmarking model latency for a production application?**
A: Benchmark across multiple dimensions: (1) Model selection — smaller models (Haiku, Titan Express) are faster than larger models (Sonnet); (2) Output length — set maxTokens to the minimum needed for your use case, as generation time scales with output tokens; (3) Throughput mode — provisioned throughput provides more consistent latency than on-demand; (4) Concurrent load — test under realistic concurrency to identify throttling; (5) Region — deploy in the region closest to your users. Run multiple iterations to account for variance and measure both average and P99 latency.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Claude 3.5 Sonnet — Section B (batch inference, 5 prompts) | 5 calls, ~2K tokens | ~$0.02 |
| Claude 3.5 Sonnet — Section C (cache misses, 2 prompts) | 2 calls, ~1K tokens | ~$0.01 |
| Titan Embeddings V2 — Section C (cache embeddings, 5 queries) | 5 calls, ~500 tokens | < $0.01 |
| Claude 3.5 Sonnet — Section D (verbose vs concise, 2 prompts) | 2 calls, ~1K tokens | ~$0.01 |
| Claude 3.5 Haiku — Section D (tiering, 2 prompts) | 2 calls, ~500 tokens | < $0.01 |
| Claude 3.5 Sonnet — Section D (tiering, 2 prompts) | 2 calls, ~1K tokens | ~$0.01 |
| Claude 3.5 Sonnet — Section E (latency, 3 runs) | 3 calls, ~600 tokens | ~$0.01 |
| Claude 3.5 Haiku — Section E (latency, 3 + 4 runs) | 7 calls, ~1.4K tokens | < $0.01 |
| Titan Text Express — Section E (latency, 3 runs) | 3 calls, ~600 tokens | < $0.01 |
| **Total (without batch job)** | | **~$0.10** |
| **Total (with batch job running)** | | **~$3-5** |

The primary cost driver is the batch inference job (Section B), which processes 5 prompts through Claude Sonnet. The on-demand API calls across all other sections are minimal. If the batch job fails due to IAM permissions, the total cost drops to approximately $0.10. Semantic caching (Section C) and model tiering (Section D) demonstrate the cost optimization strategies that would produce the largest savings in a production deployment.